# Pure Python Analysis of Portfolio Risk Metrics: VaR & CVaR Framework

## 1. Theoretical Note on Tail Risk Metrics

In financial engineering and systematic portfolio management, evaluating linear risk metrics (such as volatility or tracking error) is insufficient when asset returns exhibit non-Gaussian features, fat tails, or negative skewness. To quantify potential losses in extreme market regimes, we utilize asymmetric downside risk metrics.

### 1.1 Value at Risk (VaR)
The Value at Risk ($VaR$) at a given confidence level $\alpha \in (0, 1)$ represents the maximum expected loss over a specific time horizon $T$ that will not be exceeded with probability $\alpha$. Formally, if $\Delta V_T$ is the change in portfolio value, $VaR_\alpha$ is the negative $\left(1-\alpha\right)$-quantile of the return distribution:

$$VaR_\alpha(X) = -\inf \{ x \in \mathbb{R} : F_X(x) \ge 1 - \alpha \}$$

Under a **Parametric (Parametric Gaussian) framework**, assuming daily log-returns $r_t \sim \mathcal{N}(\mu, \sigma^2)$, the daily $VaR_\alpha$ is structured as:

$$VaR_\alpha = -(\mu + z_{1-\alpha} \cdot \sigma)$$

Where $z_{1-\alpha}$ is the standard normal quantile corresponding to the target significance level.

Under a **Historical Simulation framework**, VaR is calculated non-parametrically by sorting observed historical returns and isolating the empirical quantile directly, making no assumptions regarding the underlying distribution.

### 1.2 Conditional Value at Risk (CVaR / Expected Shortfall)
While VaR represents a specific threshold, it lacks subadditivity and fails to capture the severity of losses beyond that threshold. To satisfy the axioms of a coherent risk measure, we implement **Conditional Value at Risk (CVaR)**, which measures the expected loss conditional on the loss exceeding the VaR limit:

$$CVaR_\alpha(X) = -E[X \mid X \le -VaR_\alpha]$$

For continuous distributions, this can be formalized as the integral average of VaR levels across tail horizons:

$$CVaR_\alpha(X) = \frac{1}{1-\alpha} \int_{\alpha}^{1} VaR_u(X) \, du$$

## 2. Vectorized Multi-Asset Ingestion via yfinance
This cell downloads daily historical Adjusted Close prices for a multi-asset basket using the Yahoo Finance API, computes log-returns across all ticker dimensions simultaneously, and aggregates them into an equally-weighted portfolio vector using vectorized Pandas logic.

In [1]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats

# 1. Pure Python Ingestion via Yahoo Finance
tickers = ['AAPL', 'MSFT', 'SPY', 'TLT']
print(f"Downloading historical data for: {tickers}")
raw_prices = yf.download(tickers, start='2018-01-01', end='2026-01-01')['Adj Close']

# 2. Vectorized calculation of daily multi-asset Log Returns
df_asset_returns = np.log(raw_prices / raw_prices.shift(1)).dropna()

# 3. Portfolio Synthesis: Equally-Weighted (1/N) linear aggregation
weights = np.array([1 / len(tickers)] * len(tickers))
df_portfolio = pd.DataFrame(index=df_asset_returns.index)
df_portfolio['portfolio_log_return'] = df_asset_returns.dot(weights)

print("--- Data Pipeline Completed Successfully (No SQL Utilized) ---")
display(df_portfolio.tail())

## 3. Parametric vs. Historical Risk Estimation Engine
We encapsulate our quantitative risk calculations into an object-oriented Python structure to compute Parametric Gaussian and Historical Simulation metrics sequentially on the portfolio returns vector.

In [2]:
class PortfolioRiskEngine:
    """
    Calculates asymmetric tail-risk metrics (VaR, CVaR) for financial asset return series.
    """
    def __init__(self, returns_series: pd.Series):
        self.returns = returns_series.dropna()
        self.mu = self.returns.mean()
        self.sigma = self.returns.std()
        
    def compute_parametric_metrics(self, confidence_level: float = 0.95):
        alpha = 1 - confidence_level
        z_score = stats.norm.ppf(alpha)
        
        # Gaussian Parametric Calculations
        var_param = -(self.mu + z_score * self.sigma)
        
        # Coherent formula for Gaussian Expected Shortfall (CVaR)
        cvar_param = -(self.mu - self.sigma * (stats.norm.pdf(z_score) / alpha))
        return var_param, cvar_param
        
    def compute_historical_metrics(self, confidence_level: float = 0.95):
        alpha = 1 - confidence_level
        
        # Non-parametric empirical quantile isolation
        var_hist = -np.percentile(self.returns, alpha * 100)
        
        # Expected loss conditional on breaching the historical VaR threshold
        tail_losses = self.returns[self.returns <= -var_hist]
        cvar_hist = -tail_losses.mean()
        return var_hist, cvar_hist

# Initialize Engine with the computed portfolio log-returns
risk_engine = PortfolioRiskEngine(df_portfolio['portfolio_log_return'])

conf_level = 0.95
v_param, c_param = risk_engine.compute_parametric_metrics(confidence_level=conf_level)
v_hist, c_hist = risk_engine.compute_historical_metrics(confidence_level=conf_level)

# Tabulate metrics for industrial validation comparison
metrics_table = pd.DataFrame({
    "Parametric Gaussian": [f"{v_param*100:.4f}%", f"{c_param*100:.4f}%"],
    "Historical Simulation": [f"{v_hist*100:.4f}%", f"{c_hist*100:.4f}%"]
}, index=["Value at Risk (VaR)", "Conditional VaR (CVaR)"])

print(f"--- Systematic Portfolio Tail Risk Assessment (Confidence Level: {conf_level*100}%) ---")
display(metrics_table)

## 4. Empirical Visualization of Downside Risk
We finalize the module by generating an academic-grade empirical distribution plot using Matplotlib. This highlights the return frequencies and visualizes the tail risk regions to display potential loss thresholds.

In [3]:
fig, ax = plt.subplots(figsize=(14, 7))

# Plot the empirical portfolio return distribution histogram
count, bins, patches = ax.hist(df_portfolio['portfolio_log_return'], bins=100, density=True, 
                               color='whitesmoke', edgecolor='silver', label='Daily Portfolio Log Returns')

# Overplot theoretical Gaussian probability density function for tail comparison
x_axis = np.linspace(df_portfolio['portfolio_log_return'].min(), df_portfolio['portfolio_log_return'].max(), 500)
ax.plot(x_axis, stats.norm.pdf(x_axis, risk_engine.mu, risk_engine.sigma), 
        color='midnightblue', linewidth=2, linestyle='-', label='Theoretical Normal PDF')

# Vectorized coloring for the historical extreme risk breach zone
for patch in patches:
    if patch.get_x() <= -v_hist:
        patch.set_facecolor('mistyrose')
        patch.set_edgecolor('lightcoral')

# Vertical markers indicating empirical tail risk boundaries
ax.axvline(-v_hist, color='crimson', linewidth=2, linestyle='--', 
           label=f'Historical VaR {conf_level*100:.0f}% ({ -v_hist*100:.3f}%)')
ax.axvline(-c_hist, color='darkred', linewidth=2.5, linestyle=':', 
           label=f'Historical CVaR {conf_level*100:.0f}% ({-c_hist*100:.3f}%)')

# Academic formatting parameters
ax.set_title('Empirical Portfolio Log-Return Distribution vs. Asymmetric Tail Risk Thresholds\n(Pure Vectorized Python Pipeline via yfinance)', 
             fontweight='bold', fontsize=13, pad=15)
ax.set_xlabel('Daily Logarithmic Returns', fontweight='bold')
ax.set_ylabel('Probability Density Frequency', fontweight='bold')
ax.grid(True, linestyle='--', alpha=0.5)
ax.legend(loc='upper right', frameon=True, shadow=True, facecolor='white')

plt.tight_layout()
plt.show()